# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# Signal 1: does CTR actually fall as position gets worse?
# this is the assumption behind FlyRank's CTR-fix logic -- if it doesn't hold, comparing
# pages against an "expected ctr for this position" benchmark is pointless
visible = df[df["impressions_90d"] >= 500]
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

sig1 = (
    visible[visible["position_tier"].isin(tier_order)]
    .groupby("position_tier")["ctr"]
    .agg(["median", "mean", "count"])
    .reindex(tier_order)
)
print(sig1)

               median      mean  count
position_tier                         
top_3            0.20  0.346572    458
page_1           0.24  0.338808   7064
striking         0.17  0.266798   4485
page_3_5         0.09  0.143236   4330
deep             0.00  0.043213    389


In [3]:
# verdict: CONFIRMED. ctr drops almost the whole way down the tiers (page_1 0.24 -> striking
# 0.17 -> page_3_5 0.09 -> deep 0.00). top_3 dips slightly below page_1 (0.20 vs 0.24) but
# n=458 there is the smallest bucket -- reading that as noise, not a real reversal.

In [4]:
# Signal 2: does staleness track with worse position?
# this is the assumption behind the refresh flags -- if untouched pages don't actually rank
# worse, "it's old" isn't a reason to review it
freshness_order = ["0-30", "31-90", "91-180", "181+"]

sig2 = (
    visible.groupby("freshness_tier")["avg_position"]
    .agg(["median", "mean", "count"])
    .reindex(freshness_order)
)
print(sig2)

                median       mean  count
freshness_tier                          
0-30              10.6  15.069969  10063
31-90             12.9  16.475000     88
91-180            13.1  16.869320   6558
181+              18.6  20.670588     17


In [5]:
# verdict: CONFIRMED. median position gets worse the staler a page is: 10.6 -> 12.9 -> 13.1
# -> 18.6, in order. the middle buckets are thin (n=88 and n=17) so I'm not betting the whole
# rule on the exact numbers there, but the direction holds and 91-180 (n=6558) is solid.

**The rule, in plain words:** if a page is visible (real impressions), sitting on page 1 or
better, hasn't been touched in 90+ days, and gets fewer clicks than other pages at the same
position tier usually get — someone should look at it. Both signals above checked out, so
I'm using them together: staleness is the gate ("is this even worth someone's time"), and the
CTR gap against its own position tier is the reason ("what's actually wrong with it").

Reason code: `stale_ctr_gap`. Action label: `review_for_refresh`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import os

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

# expected ctr = the median ctr that OTHER visible pages get at the same position tier
expected_ctr = (
    df[(df["impressions_90d"] >= 500) & (df["position_tier"].isin(tier_order))]
    .groupby("position_tier")["ctr"]
    .median()
)
df["expected_ctr"] = df["position_tier"].map(expected_ctr)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

stale = (df["days_since_last_update"] >= 90).astype(int)
is_visible = (df["impressions_90d"] >= 500).astype(int)
has_position = df["position_tier"].isin(tier_order).astype(int)

# readable on purpose: zero out anything that isn't stale, visible, and ranked, then weight
# the underperformers by how much traffic they're actually wasting
df["score"] = stale * is_visible * has_position * df["ctr_gap"] * df["impressions_90d"]
df["reason_code"] = "stale_ctr_gap"
df["action_label"] = "review_for_refresh"

print("flagged (score > 0):", (df["score"] > 0).sum(), "of", len(df))

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "reason_code", "action_label",
     "impressions_90d", "days_since_last_update", "position_tier",
     "ctr", "expected_ctr", "ctr_gap"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
queue.head(10)

flagged (score > 0): 3227 of 30000


,content_id,client_id,score,reason_code,action_label,impressions_90d,days_since_last_update,position_tier,ctr,expected_ctr,ctr_gap
3394,content_36ff89c8214e,client_19581e27de,56068.43,stale_ctr_gap,review_for_refresh,295097,104,page_1,0.05,0.24,0.19
6653,content_5fe46e04994d,client_4e07408562,51771.50,stale_ctr_gap,review_for_refresh,517715,104,page_1,0.14,0.24,0.10
7445,content_c8e9d6ab9013,client_19581e27de,50082.72,stale_ctr_gap,review_for_refresh,208678,104,page_1,0.00,0.24,0.24
3070,content_91652435f57a,client_19581e27de,28726.20,stale_ctr_gap,review_for_refresh,159590,104,page_1,0.06,0.24,0.18
9193,content_c1fe78bc4e37,client_19581e27de,28151.55,stale_ctr_gap,review_for_refresh,134055,104,page_1,0.03,0.24,0.21
26474,content_a7427266c305,client_19581e27de,26144.43,stale_ctr_gap,review_for_refresh,201111,104,page_1,0.11,0.24,0.13
4708,content_b115f7c74779,client_19581e27de,25928.49,stale_ctr_gap,review_for_refresh,123469,104,page_1,0.03,0.24,0.21
5621,content_97a86caf3a3d,client_19581e27de,25103.90,stale_ctr_gap,review_for_refresh,147670,104,page_1,0.07,0.24,0.17
26531,content_cb112fce36be,client_19581e27de,24792.80,stale_ctr_gap,review_for_refresh,309910,104,page_1,0.16,0.24,0.08
3331,content_4a6607efcb46,client_6208ef0f77,24332.92,stale_ctr_gap,review_for_refresh,128068,104,top_3,0.01,0.20,0.19


## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

1. **content_36ff89c8214e** (client_19581e27de) — page_1, 295k impressions, ctr 0.05 vs 0.24
   expected, untouched 104 days. Wrong if this client's canonical/URL setup is splitting one
   page's clicks across variants — the gap would be a tracking artifact, not a content problem.
2. **content_5fe46e04994d** (client_4e07408562) — page_1, 517k impressions (the biggest one
   here), ctr 0.14 vs 0.24. Wrong if the query mix behind it is mostly brand/navigational —
   people already know what they're clicking, no title rewrite fixes that.
3. **content_c8e9d6ab9013** (client_19581e27de) — page_1, 208k impressions, ctr basically 0.00.
   Wrong if GSC tracking is actually broken on this page — 0 clicks on 208k impressions is
   suspicious enough that I'd check the tag before touching the content.
4. **content_91652435f57a** (client_19581e27de) — page_1, 159k impressions, ctr 0.06. Wrong if
   this is the same template/URL family as #1, #3, #5-#8 below — one metadata fix could resolve
   all of them, meaning this isn't really 6 separate problems.
5. **content_c1fe78bc4e37** (client_19581e27de) — page_1, 134k impressions, ctr 0.03. Wrong if
   the query behind it had a one-off demand spike or dip — a temporary CTR dent, not a lasting one.
6. **content_a7427266c305** (client_19581e27de) — page_1, 201k impressions, ctr 0.11. Wrong if
   the page_1 "expected" benchmark (0.24) is being pulled up by a different intent mix than this
   page serves — comparing it against the wrong peer group.
7. **content_b115f7c74779** (client_19581e27de) — page_1, 123k impressions, ctr 0.03. Same
   client-concentration risk as #1/#3/#4/#5/#6 — see the weak-picks note below.
8. **content_97a86caf3a3d** (client_19581e27de) — page_1, 147k impressions, ctr 0.07. Same risk.
9. **content_cb112fce36be** (client_19581e27de) — page_1, 309k impressions, ctr 0.16, the
   smallest gap of the group (0.08). Wrong if this one's just normal noise around the median —
   it's the weakest pick in this top 10.
10. **content_4a6607efcb46** (client_6208ef0f77) — top_3, 128k impressions, ctr 0.01 vs 0.20.
    Wrong if a featured snippet or answer box is sitting on top of this query — that eats clicks
    structurally, no metadata change reclaims it.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks.** 8 of the top 10 belong to `client_19581e27de`, and all of them show
`days_since_last_update == 104` exactly. That's not 8 independently-earned signals — it reads
like one batch content-update event on that client's site, and the rule has no way to tell the
difference between "8 real problems" and "1 client-wide thing that happened to touch 8 pages."
#9 (`content_cb112fce36be`) is also the weakest pick on its own merits — its ctr_gap (0.08) is
the smallest in the list, closest to just being normal spread around the tier median rather
than a real underperformer. If I were actually working this queue I'd cap it at 1-2 pages per
client per run, or de-dup on `days_since_last_update` ties, before trusting the ranking at scale.

**Leakage check.** The score only touches `impressions_90d`, `days_since_last_update`,
`position_tier`, and `ctr` — all observed over the same trailing 90-day window, none of them a
future window. `trend_direction` and `trend_pct` (the label source) never enter the score, and
this dataset doesn't ship any FlyRank product flag (`health_score`, `priority_score`, etc.) to
begin with, so there's nothing there to accidentally leak in.

In [7]:
# concentration check backing the weak-picks note above
print(queue.head(10)["client_id"].value_counts())

# leakage check: score-building columns vs the banned label-source columns
score_inputs = {"impressions_90d", "days_since_last_update", "position_tier", "ctr"}
label_source = {"trend_direction", "trend_pct"}
print("banned columns touched by the score:", score_inputs & label_source)

client_id
client_19581e27de    8
client_4e07408562    1
client_6208ef0f77    1
Name: count, dtype: int64
banned columns touched by the score: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.